# Auditoría de Calidad de Datos - NovaMarket
**Archivo:** `S03_PD_Martinez_Sandoval_Torres_Olmo_AuditoriaCalidad.ipynb`

**Dataset:** `NovaMarket_datos_crudos.csv`

**Entrega grupal**

**Equipo:**
- Diego Alejandro Sandoval
- Santiago Torres
- Juan Andrés Martínez
- Laura Valentina Olmos

Este cuaderno audita el dataset crudo de NovaMarket en las **6 dimensiones de calidad de datos**: completitud, exactitud, consistencia, validez, unicidad y oportunidad. Cada hallazgo se cuantifica con conteos exactos, y al final se resume en un catálogo de problemas priorizado por severidad (`S03_PD_Martinez_Sandoval_Torres_Olmo_CatalogoProblemas.csv`).

In [1]:
import pandas as pd

df = pd.read_csv('NovaMarket_datos_crudos.csv')
print(df.shape)
df.head()

(620, 14)


,id_pedido,id_cliente,fecha_compra,canal,ciudad,codigo_postal,categoria_producto,producto,precio,unidades,edad_cliente,correo,nivel_satisfaccion,fecha_actualizacion_stock
0,P00383,C0276,2026-01-03,App,Bucaramanga,68001,Belleza,Camiseta,-1742611.0,4,-19,c0276gmail.com,alto,2026-07-01
1,P00403,C0167,6/12/2026,Web,Cali,76001,Deportes,Cafetera,1828583.0,2,23,c0167@hotmail.com,MEDIO,2026-07-01
2,P00403,C0167,6/12/2026,Web,Cali,76001,Deportes,Cafetera,1828583.0,2,23,c0167@hotmail.com,MEDIO,2026-07-01
3,P00450,C0128,07/09/2026,App,bucaramanga,68001,electronica,Set bloques,787178.0,6,69,c0128@hotmail.com,MEDIO,2026-07-15
4,P00449,C0304,2025-07-19,Tienda,Bogotá,8001,Belleza,Audífonos BT,357231.0,5,67,c0304@novamarket.co,Bajo,2026-05-20


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 620 entries, 0 to 619
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id_pedido                  620 non-null    object 
 1   id_cliente                 620 non-null    object 
 2   fecha_compra               620 non-null    object 
 3   canal                      620 non-null    object 
 4   ciudad                     589 non-null    object 
 5   codigo_postal              620 non-null    int64  
 6   categoria_producto         620 non-null    object 
 7   producto                   620 non-null    object 
 8   precio                     599 non-null    float64
 9   unidades                   620 non-null    int64  
 10  edad_cliente               620 non-null    int64  
 11  correo                     516 non-null    object 
 12  nivel_satisfaccion         574 non-null    object 
 13  fecha_actualizacion_stock  620 non-null    object 

---
## 1. Auditoría de Completitud

Revisamos cuántos valores faltantes (`NaN`) tiene cada columna, en cantidad y en porcentaje sobre el total de **620** registros.

In [3]:
df.isnull().sum()

id_pedido                      0
id_cliente                     0
fecha_compra                   0
canal                          0
ciudad                        31
codigo_postal                  0
categoria_producto             0
producto                       0
precio                        21
unidades                       0
edad_cliente                   0
correo                       104
nivel_satisfaccion            46
fecha_actualizacion_stock      0
dtype: int64

In [4]:
(df.isnull().mean() * 100).round(2)

id_pedido                     0.00
id_cliente                    0.00
fecha_compra                  0.00
canal                         0.00
ciudad                        5.00
codigo_postal                 0.00
categoria_producto            0.00
producto                      0.00
precio                        3.39
unidades                      0.00
edad_cliente                  0.00
correo                       16.77
nivel_satisfaccion            7.42
fecha_actualizacion_stock     0.00
dtype: float64

In [5]:
# Revisar si hay "falsos completos" en columnas categóricas clave (valores como "N/A" o "-")
df["ciudad"].value_counts(dropna=False)

ciudad
bucaramanga     58
CARTAGENA       52
Cartagena       50
Bucaramanga     47
cali            39
CALI            36
barranquilla    36
medellin        32
NaN             31
Cali            29
Barranquilla    29
MEDELLIN        26
Medellin        25
B/quilla        23
Bogota          21
BOGOTÁ          21
Bogotá          20
Bogotá D.C.     17
bogota          14
Medellín        14
Name: count, dtype: int64

In [6]:
df["nivel_satisfaccion"].value_counts(dropna=False)

nivel_satisfaccion
Alto     91
ALTO     84
Medio    74
bajo     73
medio    70
alto     67
Bajo     62
MEDIO    53
NaN      46
Name: count, dtype: int64

## **Hallazgos de completitud:**

- `correo`: **104** valores faltantes (**16.77%** del total). Es el campo más incompleto: sin correo no se puede contactar directamente a 1 de cada 6 clientes.
- `nivel_satisfaccion`: **46** valores faltantes (**7.42%**).
- `ciudad`: **31** valores faltantes (**5.00%**).
- `precio`: **21** valores faltantes (**3.39%**). Aunque el porcentaje es bajo, es un campo financiero clave: sin precio no se puede calcular el ingreso de esos pedidos.
- El resto de columnas no presentan valores faltantes.
- Al revisar `ciudad` y `nivel_satisfaccion` con `value_counts()`, no se encontraron "falsos completos" (no aparecen valores como "N/A", "-" o "Sin dato"); los faltantes son `NaN` reales.

---
## 2. Auditoría de Exactitud

Aplicamos reglas de dominio sobre `precio`, `unidades` y `edad_cliente` para encontrar valores que, aunque tengan el tipo de dato correcto, son imposibles en el contexto del negocio.

In [7]:
# Regla 1: precio negativo
precio_negativo = df[df["precio"] < 0]
print("Precio negativo:", len(precio_negativo))

# Regla 2: precio absurdamente alto
precio_absurdo = df[df["precio"] > 10_000_000]
print("Precio > $10.000.000:", len(precio_absurdo))
precio_absurdo[["producto", "precio"]]

Precio negativo: 19
Precio > $10.000.000: 6


,producto,precio
71,Zapatillas,142211382.0
399,Cafetera,773003787.0
442,Zapatillas,253768505.0
541,Perfume,521536977.0
566,Aspiradora,218025714.0
580,Cafetera,336563515.0


In [8]:
# Regla 3: unidades negativas
unidades_negativas = df[df["unidades"] < 0]
print("Unidades negativas:", len(unidades_negativas))

# Regla 4: unidades en cero
unidades_cero = df[df["unidades"] == 0]
print("Unidades en cero:", len(unidades_cero))

# Regla 5 (propia del equipo): unidades absurdas, más de 100 en una sola línea de pedido
unidades_absurdas = df[df["unidades"] > 100]
print("Unidades > 100 en una sola línea:", len(unidades_absurdas))
unidades_absurdas[["producto", "unidades"]].sort_values("unidades", ascending=False)

Unidades negativas: 14
Unidades en cero: 17
Unidades > 100 en una sola línea: 15


,producto,unidades
26,Muñeca,9999
84,Aspiradora,9999
62,Zapatillas,9999
102,Audífonos BT,9999
526,Mancuernas,9999
291,Smartphone,9999
251,Crema facial,9999
392,Zapatillas,9999
226,Crema facial,5000
9,Cafetera,5000


In [9]:
# Regla 6: edad negativa
edad_negativa = df[df["edad_cliente"] < 0]
print("Edad negativa:", len(edad_negativa))

# Regla 7 (propia del equipo): edad mayor a 120 años, imposible
edad_absurda = df[df["edad_cliente"] > 120]
print("Edad > 120 años:", len(edad_absurda))
print("Valores encontrados:", sorted(edad_absurda["edad_cliente"].unique()))

Edad negativa: 15
Edad > 120 años: 17
Valores encontrados: [np.int64(150), np.int64(200)]


## **Hallazgos de exactitud** (7 reglas de dominio aplicadas):

- `precio`: **19** registros con precio negativo (**3.06%**) — imposible en una venta, distorsiona cualquier suma de ingresos.
- `precio`: **6** registros con precio mayor a $10.000.000 (**0.97%**) — muy por fuera del catálogo real de NovaMarket.
- `unidades`: **14** registros con cantidad negativa (**2.26%**).
- `unidades`: **17** registros con cantidad en cero (**2.74%**) — un pedido sin unidades no debería existir.
- `unidades`: **15** registros con cantidades absurdas (5.000 o 9.999 unidades en una sola línea) (**2.42%**). **Regla propia del equipo:** ninguna línea de pedido individual debería superar las 100 unidades.
- `edad_cliente`: **15** registros con edad negativa (**2.42%**).
- `edad_cliente`: **17** registros con edad mayor a 120 años, con valores de 150 y 200 (**2.74%**). **Regla propia del equipo:** ningún cliente puede tener más de 120 años.

---
## 3. Auditoría de Consistencia

Con `value_counts()` revisamos si el mismo valor real está escrito de formas distintas (mayúsculas, tildes, abreviaturas) en tres columnas categóricas, y sumamos manualmente los registros de las variantes que representan el mismo valor real.

In [10]:
df["ciudad"].value_counts(dropna=False)

ciudad
bucaramanga     58
CARTAGENA       52
Cartagena       50
Bucaramanga     47
cali            39
CALI            36
barranquilla    36
medellin        32
NaN             31
Cali            29
Barranquilla    29
MEDELLIN        26
Medellin        25
B/quilla        23
Bogota          21
BOGOTÁ          21
Bogotá          20
Bogotá D.C.     17
bogota          14
Medellín        14
Name: count, dtype: int64

Agrupando manualmente las variantes que representan la misma ciudad:

- **Bogotá** (`Bogotá` + `BOGOTÁ` + `Bogota` + `bogota` + `Bogotá D.C.`) = **93** registros
- **Barranquilla** (`Barranquilla` + `barranquilla` + `B/quilla`) = **88** registros
- **Bucaramanga** = **105** registros (sin variantes)
- **Cali** = **104** registros (sin variantes)
- **Cartagena** = **102** registros (sin variantes)
- **Medellín** (`Medellín` + `medellin` + `MEDELLIN`) = **97** registros

En total, `ciudad` tiene **19 valores de texto distintos** que en realidad representan solo **6 ciudades reales**.

In [11]:
df["categoria_producto"].value_counts(dropna=False)

categoria_producto
Belleza        66
deportes       60
Deportes       59
belleza        54
Moda           39
Juguetería     37
MODA           35
HOGAR          34
Hogar          33
Electronica    33
hogar          32
juguetes       30
Jugueteria     28
moda           24
Electrónica    21
electronica    18
ELECTRONICA    17
Name: count, dtype: int64

Agrupando manualmente las variantes que representan la misma categoría:

- **Belleza** = **120** registros
- **Deportes** = **119** registros
- **Hogar** = **99** registros
- **Moda** = **98** registros
- **Electrónica** (`Electrónica` + `electronica`) = **89** registros
- **Juguetería** (`Juguetería` + `jugueteria` + `juguetes`) = **95** registros

En total, `categoria_producto` tiene **17 valores de texto distintos** que representan solo **6 categorías reales**.

In [12]:
df["nivel_satisfaccion"].value_counts(dropna=False)

nivel_satisfaccion
Alto     91
ALTO     84
Medio    74
bajo     73
medio    70
alto     67
Bajo     62
MEDIO    53
NaN      46
Name: count, dtype: int64

Agrupando manualmente las variantes que representan el mismo nivel:

- **Alto** (`Alto` + `ALTO` + `alto`) = **242** registros
- **Medio** (`Medio` + `MEDIO` + `medio`) = **197** registros
- **Bajo** (`Bajo` + `BAJO`) = **135** registros
- Faltantes (`NaN`) = **46** registros (ya contados en la sección de completitud)

En total, `nivel_satisfaccion` tiene **8 valores de texto distintos** (sin contar `NaN`) que representan solo **3 niveles reales**.

## **Hallazgos de consistencia:**

- `ciudad`: **19** valores de texto distintos representan solo **6** ciudades reales. Bucaramanga (105), Cali (104) y Cartagena (102) no tienen variantes; Bogotá (93), Barranquilla (88) y Medellín (97) sí, por mayúsculas, tildes o abreviaturas.
- `categoria_producto`: **17** valores de texto distintos representan solo **6** categorías reales. Belleza (120), Deportes (119), Hogar (99) y Moda (98) no tienen variantes; Electrónica (89) y Juguetería (95) sí, por mayúsculas, tildes o el uso de "juguetes" como sinónimo.
- `nivel_satisfaccion`: **8** valores de texto distintos (sin contar `NaN`) representan solo **3** niveles reales: Alto (242), Medio (197) y Bajo (135).
- En las tres columnas, el problema es el mismo: mayúsculas/minúsculas y tildes hacen que `value_counts()` y cualquier `groupby()` traten un mismo valor real como si fueran valores distintos, lo que rompe cualquier reporte agregado por ciudad, categoría o nivel de satisfacción hasta que se normalice el texto.

---
## 4. Auditoría de Validez

Revisamos si el correo tiene el símbolo `@` y si la fecha de compra se puede convertir con `pd.to_datetime(errors="coerce")`, tal como se vio en la lección de validez.

In [13]:
# Correos sin @ (solo entre los que sí tienen valor, para no mezclar con los faltantes de la sección 1)
correo_no_nulo = df["correo"].notna()
correos_invalidos = df[correo_no_nulo & ~df["correo"].str.contains("@", na=False)]

print("Correos no nulos:", correo_no_nulo.sum())
print("Correos sin @:", len(correos_invalidos))
correos_invalidos["correo"].head(10)

Correos no nulos: 516
Correos sin @: 51


0      c0276gmail.com
33     c0365gmail.com
40     c0093gmail.com
41     c0319gmail.com
44     c0310gmail.com
76     c0297gmail.com
81     c0218gmail.com
94     c0349gmail.com
96     c0391gmail.com
102    c0015gmail.com
Name: correo, dtype: object

In [14]:
# Conversión de fecha_compra con el método visto en clase
fecha_compra_conv = pd.to_datetime(df["fecha_compra"], errors="coerce")

fechas_invalidas = df[fecha_compra_conv.isna() & df["fecha_compra"].notna()]
print("Fechas de compra que no se pudieron convertir:", len(fechas_invalidas))
fechas_invalidas["fecha_compra"].head(10)

Fechas de compra que no se pudieron convertir: 401


1      6/12/2026
2      6/12/2026
3     07/09/2026
5       6/8/2025
6      13/4/2026
7     19/11/2025
8      22/6/2025
9      17/9/2026
10     9/10/2025
11    19/11/2026
Name: fecha_compra, dtype: object

In [15]:
# Fechas de compra futuras (posteriores a hoy)
hoy = pd.Timestamp.today()
fechas_futuras = df[fecha_compra_conv > hoy]

print("Fecha de referencia (hoy):", hoy.date())
print("Fechas de compra futuras:", len(fechas_futuras))
fechas_futuras[["id_pedido", "fecha_compra"]].head(10)

Fecha de referencia (hoy): 2026-08-05
Fechas de compra futuras: 44


,id_pedido,fecha_compra
18,P00273,2026-11-22
23,P00415,2026-08-19
41,P00316,2027-05-20
46,P00121,2026-11-02
49,P00284,2026-12-19
57,P00041,2026-10-22
70,P00104,2026-10-20
85,P00177,2026-09-04
99,P00376,2026-11-26
120,P00560,2026-09-11


## **Hallazgos de validez:**

- `correo`: de los **516** correos no nulos, **51** (**9.88%** de los no nulos, **8.23%** del total) no contienen el símbolo `@`, por ejemplo `c0276gmail.com`.
- `fecha_compra`: al convertir con `pd.to_datetime(errors="coerce")`, **401** registros (**64.68%**) no se pudieron convertir. Al revisar los ejemplos, notamos que la mayoría **no son fechas realmente inexistentes**, sino fechas válidas escritas en formato `D/M/AAAA` (ej. `6/12/2026`) que `pandas` no reconoció porque el resto de la columna está en formato `AAAA-MM-DD` y el método simple asume un solo formato para toda la columna. Esto confirma, desde otro ángulo, el problema de **consistencia de formato** de fechas que ya veníamos sospechando: la columna mezcla dos formatos distintos y eso rompe la conversión automática.
- `fecha_compra`: de las fechas que sí se lograron convertir, **44** (**7.10%**) quedan como posteriores a hoy — probablemente hay más fechas futuras "escondidas" entre las 401 no convertidas, así que este número es un piso, no el total real.

---
## 5. Auditoría de Unicidad

Buscamos filas 100% duplicadas y duplicados en `id_pedido`, que debería ser una clave única, con `duplicated()`.

In [16]:
# Filas completamente duplicadas
print("Filas 100% duplicadas:", df.duplicated().sum())

Filas 100% duplicadas: 20


In [17]:
# Duplicados por id_pedido (clave que debería ser única)
duplicados_id = df[df.duplicated(subset=["id_pedido"], keep=False)].sort_values("id_pedido")

print("Filas involucradas en algún id_pedido duplicado:", len(duplicados_id))
duplicados_id[["id_pedido", "id_cliente", "fecha_compra", "producto"]].head(6)

Filas involucradas en algún id_pedido duplicado: 40


,id_pedido,id_cliente,fecha_compra,producto
380,P00030,C0099,2027-05-20,Set bloques
240,P00030,C0099,2027-05-20,Set bloques
554,P00043,C0012,4/11/2025,Mancuernas
308,P00043,C0012,4/11/2025,Mancuernas
585,P00110,C0265,2026-01-23,Smart TV
294,P00110,C0265,2026-01-23,Smart TV


## **Hallazgos de unicidad:**

- **20** filas están 100% duplicadas: todas sus columnas son idénticas a otra fila (**3.23%**).
- `id_pedido` tiene **20** valores repetidos, lo que involucra **40** filas en total (**6.45%**). Al revisar manualmente algunas de esas filas, confirmamos que corresponden al mismo pedido reingresado dos veces, no a coincidencias.

---
## 6. Auditoría de Oportunidad

Calculamos los días transcurridos desde `fecha_actualizacion_stock` hasta hoy, y definimos un umbral de negocio para considerar el stock "desactualizado".

In [18]:
fecha_actualizacion_conv = pd.to_datetime(df["fecha_actualizacion_stock"], errors="coerce")
dias_desde_actualizacion = (hoy - fecha_actualizacion_conv).dt.days

dias_desde_actualizacion.describe()

count    620.000000
mean     130.930645
std      276.890780
min     -208.000000
25%       21.000000
50%       56.000000
75%       77.000000
max      979.000000
Name: fecha_actualizacion_stock, dtype: float64

**¿Por qué un umbral de 90 días?** Con `describe()` vemos que el 75% de los registros lleva 77 días o menos sin actualizarse, pero el máximo es de 979 días (casi 3 años). Ese salto tan grande entre el percentil 75 (77 días) y el máximo (979 días) muestra que hay dos grupos bien separados: la mayoría del inventario se actualiza con frecuencia, y un grupo aparte lleva mucho tiempo sin tocarse. Por eso 90 días es un umbral razonable y no arbitrario para marcar el stock como desactualizado.

In [19]:
umbral_dias = 90
stock_desactualizado = df[dias_desde_actualizacion > umbral_dias]

print(f"Registros con stock sin actualizar hace más de {umbral_dias} días:", len(stock_desactualizado))
stock_desactualizado[["producto", "fecha_actualizacion_stock"]].head(10)

Registros con stock sin actualizar hace más de 90 días: 67


,producto,fecha_actualizacion_stock
6,Audífonos BT,2024-01-15
13,Smart TV,2024-06-01
17,Zapatillas,2024-06-01
25,Smartphone,2024-06-01
47,Set bloques,2024-01-15
49,Camiseta,2024-01-15
51,Balón fútbol,2024-01-15
61,Cafetera,2024-01-15
69,Smart TV,2024-01-15
86,Smart TV,2023-11-30


In [20]:
# Fechas de actualización de stock en el futuro (imposible)
stock_fecha_futura = df[fecha_actualizacion_conv > hoy]

print("Registros con fecha_actualizacion_stock en el futuro:", len(stock_fecha_futura))
stock_fecha_futura[["producto", "fecha_actualizacion_stock"]].head(10)

Registros con fecha_actualizacion_stock en el futuro: 21


,producto,fecha_actualizacion_stock
7,Cafetera,2027-03-01
56,Camiseta,2027-03-01
87,Smart TV,2027-03-01
96,Muñeca,2027-03-01
99,Aspiradora,2027-03-01
106,Smart TV,2027-03-01
205,Crema facial,2027-03-01
222,Cafetera,2027-03-01
258,Aspiradora,2027-03-01
289,Muñeca,2027-03-01


## **Hallazgos de oportunidad:**

- **67** registros (**10.81%**) llevan más de **90 días** sin actualizar su stock.
- **21** registros (**3.39%**) tienen `fecha_actualizacion_stock` en el futuro, lo cual es imposible: el stock no puede haberse actualizado en una fecha que todavía no llega.

---
## 7. Resumen ejecutivo y catálogo de problemas

Consolidamos todos los hallazgos anteriores en el catálogo `S03_PD_Martinez_Sandoval_Torres_Olmo_CatalogoProblemas.csv`, ordenado de mayor a menor severidad.

**Criterio de severidad** (combina dos factores, no es solo el porcentaje):
- **Crítico**: el error hace que el dato sea lógica o financieramente imposible (ej. precio negativo), o afecta a más del 15% del dataset.
- **Alto**: afecta entre el 5% y el 15% del dataset, o un porcentaje menor pero en un campo financiero/operativo clave (precio, unidades, id_pedido).
- **Medio**: afecta entre 1% y 5% del dataset en un campo no financiero, o son variantes de texto que degradan el reporting pero no rompen cálculos.
- **Bajo**: afecta a menos del 1% del dataset y tiene bajo impacto operativo.

In [21]:
catalogo = [
    ("Validez", "fecha_compra", "Fecha que no se pudo convertir con pd.to_datetime (mezcla de formatos + fechas inexistentes)", 401, 64.68, "Crítico",
     "Afecta a la mayoría del dataset y compromete cualquier análisis temporal o de ingresos por periodo."),
    ("Exactitud", "precio", "Precio negativo", 19, 3.06, "Crítico",
     "Campo financiero directo; un precio negativo es lógicamente imposible."),
    ("Consistencia", "ciudad", "Variantes del mismo valor (mayúsculas, tildes, abreviaturas): 19 valores para 6 ciudades reales", 400, 64.52, "Alto",
     "Sin normalizar, cualquier segmentación geográfica o logística queda rota."),
    ("Consistencia", "categoria_producto", "Variantes del mismo valor: 17 valores para 6 categorías reales", 365, 58.87, "Alto",
     "Rompe el reporting de ventas por categoría, una de las vistas más usadas del negocio."),
    ("Completitud", "correo", "Valor faltante (NaN)", 104, 16.77, "Alto",
     "Bloquea la comunicación directa con 1 de cada 6 clientes."),
    ("Oportunidad", "fecha_actualizacion_stock", "Stock sin actualizar hace más de 90 días", 67, 10.81, "Alto",
     "Decisiones de inventario tomadas sobre datos de varios meses (hasta casi 3 años) de antigüedad."),
    ("Validez", "fecha_compra", "Fecha de compra posterior a hoy", 44, 7.10, "Alto",
     "Una venta no puede haber ocurrido en el futuro; probablemente es un piso, hay más casos ocultos entre las fechas no convertidas."),
    ("Unicidad", "id_pedido", "Pedido duplicado (filas involucradas)", 40, 6.45, "Alto",
     "Cada pedido duplicado infla artificialmente el conteo de ventas e ingresos en reportes agregados."),
    ("Exactitud", "unidades", "Cantidad absurda (5.000 o 9.999 unidades en una sola línea)", 15, 2.42, "Alto",
     "Aunque el porcentaje es bajo, cada outlier distorsiona fuertemente cualquier suma de ingresos o inventario."),
    ("Completitud", "precio", "Valor faltante (NaN)", 21, 3.39, "Alto",
     "Campo financiero clave: sin precio no se puede calcular el ingreso de esos pedidos."),
    ("Consistencia", "nivel_satisfaccion", "Variantes del mismo valor: 8 valores para 3 niveles reales", 347, 55.97, "Medio",
     "Afecta el análisis de satisfacción de cliente, sin comprometer cifras financieras."),
    ("Validez", "correo", "Formato inválido (sin @)", 51, 8.23, "Medio",
     "Correos no utilizables para contacto; volumen menor que el problema de completitud del mismo campo."),
    ("Completitud", "nivel_satisfaccion", "Valor faltante (NaN)", 46, 7.42, "Medio",
     "Limita el análisis de satisfacción para ese subconjunto de clientes."),
    ("Exactitud", "edad_cliente", "Edad imposible (negativa o mayor a 120 años)", 32, 5.16, "Medio",
     "Afecta la segmentación demográfica, sin impacto financiero directo."),
    ("Completitud", "ciudad", "Valor faltante (NaN)", 31, 5.00, "Medio",
     "Limita el análisis geográfico y logístico para ese subconjunto de pedidos."),
    ("Oportunidad", "fecha_actualizacion_stock", "Fecha de actualización en el futuro", 21, 3.39, "Medio",
     "El stock no puede haberse actualizado en una fecha que todavía no llega."),
    ("Exactitud", "unidades", "Cantidad en cero", 17, 2.74, "Medio",
     "Un pedido sin unidades no debería existir como venta; revisar si son cancelaciones mal registradas."),
    ("Exactitud", "unidades", "Cantidad negativa", 14, 2.26, "Medio",
     "Financieramente imposible, pero volumen bajo comparado con otros hallazgos de exactitud."),
    ("Exactitud", "precio", "Precio absurdamente alto (> $10.000.000)", 6, 0.97, "Bajo",
     "Volumen muy bajo; revisar caso a caso, posible error de dígitos de más."),
]

catalogo_df = pd.DataFrame(
    catalogo,
    columns=["dimension", "columna", "problema", "registros_afectados", "pct_afectado", "severidad", "justificacion"]
)

catalogo_df

,dimension,columna,problema,registros_afectados,pct_afectado,severidad,justificacion
0,Validez,fecha_compra,Fecha que no se pudo convertir con pd.to_datet...,401,64.68,Crítico,Afecta a la mayoría del dataset y compromete c...
1,Exactitud,precio,Precio negativo,19,3.06,Crítico,Campo financiero directo; un precio negativo e...
2,Consistencia,ciudad,"Variantes del mismo valor (mayúsculas, tildes,...",400,64.52,Alto,"Sin normalizar, cualquier segmentación geográf..."
3,Consistencia,categoria_producto,Variantes del mismo valor: 17 valores para 6 c...,365,58.87,Alto,"Rompe el reporting de ventas por categoría, un..."
4,Completitud,correo,Valor faltante (NaN),104,16.77,Alto,Bloquea la comunicación directa con 1 de cada ...
5,Oportunidad,fecha_actualizacion_stock,Stock sin actualizar hace más de 90 días,67,10.81,Alto,Decisiones de inventario tomadas sobre datos d...
6,Validez,fecha_compra,Fecha de compra posterior a hoy,44,7.10,Alto,Una venta no puede haber ocurrido en el futuro...
7,Unicidad,id_pedido,Pedido duplicado (filas involucradas),40,6.45,Alto,Cada pedido duplicado infla artificialmente el...
8,Exactitud,unidades,Cantidad absurda (5.000 o 9.999 unidades en un...,15,2.42,Alto,"Aunque el porcentaje es bajo, cada outlier dis..."
9,Completitud,precio,Valor faltante (NaN),21,3.39,Alto,Campo financiero clave: sin precio no se puede...


In [22]:
catalogo_df.to_csv("S03_PD_Martinez_Sandoval_Torres_Olmo_CatalogoProblemas.csv", index=False, encoding="utf-8-sig")
print("Catálogo guardado:", len(catalogo_df), "filas")
catalogo_df["severidad"].value_counts()

Catálogo guardado: 19 filas


severidad
Alto       8
Medio      8
Crítico    2
Bajo       1
Name: count, dtype: int64

**Conclusión general:** El dataset crudo de NovaMarket tiene problemas en las 6 dimensiones evaluadas. El hallazgo de mayor volumen es la mezcla de formatos de fecha en `fecha_compra`, que hace fallar la conversión simple en el 64.68% de los registros; junto con los precios negativos, son los dos hallazgos de severidad **crítica** que deben corregirse antes de cualquier análisis de ingresos o tendencias. Los problemas de **consistencia** en `ciudad`, `categoria_producto` y `nivel_satisfaccion` son los de mayor volumen dentro de sus columnas, pero se solucionan con un proceso de normalización de texto y agrupación manual relativamente simple. El catálogo completo con los 19 hallazgos, su severidad y justificación queda disponible en `S03_PD_Martinez_Sandoval_Torres_Olmo_CatalogoProblemas.csv`.